
# Buy Fee Model V12B — 2022 Buyer Baseline for 2023+ Fee Events

## Why this notebook

The buyer lifecycle file ends in **September 2022**, while the Buy Fee events begin in **2023**.

Therefore, a true buyer PRE/POST event study is not possible with the current buyer history.

This notebook uses the buyer data in a way that is valid:

1. Build a **historical buyer-health baseline** from 2022.
2. Aggregate it by auction location.
3. Join that historical buyer profile to Buy Fee events from **January 1, 2023 onward**.
4. Compare whether locations with stronger/weaker historical buyer health had different:
   - RPU outcomes
   - Conversion outcomes
   - Volume outcomes
5. Compare +$10 / +$15 / +$20 separately.
6. Identify which buyer baseline features are worth testing in the next predictive model.

## Important limitation

These buyer features are **historical location characteristics**, not event-time buyer behavior.

They should be interpreted as:
> "What kind of buyer ecosystem did this location historically have?"

They should **not** be interpreted as:
> "How did buyers react immediately to the 2023–2026 fee increase?"

That requires newer buyer data.


In [1]:

from pathlib import Path
import pandas as pd
import numpy as np
import gc
import warnings

warnings.filterwarnings("ignore")

# ============================================================
# CONFIGURATION
# ============================================================

DATA_FOLDER = Path(
    r"C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion"
)

BUYER_FILE = DATA_FOLDER / "biweekly_buyer_churn_flags_detail_reactivated_loop1.csv"

V82_DIR = DATA_FOLDER / "buy_fee_model_v8_2_outputs"

DID_FILE = V82_DIR / "04_DID_EVENT_LEVEL_RESULTS.csv"

OUTPUT_DIR = DATA_FOLDER / "buy_fee_model_v12b_2022baseline_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Buyer baseline window
BUYER_BASELINE_START = pd.Timestamp("2022-01-01")
BUYER_BASELINE_END = pd.Timestamp("2022-09-30")

# Fee events
FEE_EVENT_START = pd.Timestamp("2023-01-01")

SCENARIOS = [10, 15, 20]
SCENARIO_TOLERANCE = 2.5

CHUNK_SIZE = 100_000

# Balanced success definition
RPU_SUCCESS_MIN = 0.0
CONVERSION_SAFE_MIN_PP = -2.0
VOLUME_SAFE_MIN_PCT = -0.10

print("=" * 80)
print("INPUT CHECK")
print("=" * 80)

print("Buyer file:", BUYER_FILE)
print("Exists:", BUYER_FILE.exists())

print("\nDiD file:", DID_FILE)
print("Exists:", DID_FILE.exists())

if not BUYER_FILE.exists():
    raise FileNotFoundError(BUYER_FILE)

if not DID_FILE.exists():
    raise FileNotFoundError(DID_FILE)

print("\nOutput folder:")
print(OUTPUT_DIR)


INPUT CHECK
Buyer file: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\biweekly_buyer_churn_flags_detail_reactivated_loop1.csv
Exists: True

DiD file: C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v8_2_outputs\04_DID_EVENT_LEVEL_RESULTS.csv
Exists: True

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12b_2022baseline_outputs


## 1. Inspect buyer columns

In [2]:

header = pd.read_csv(
    BUYER_FILE,
    nrows=5,
    low_memory=True,
    encoding="utf-8",
    encoding_errors="replace"
)

print("Buyer columns:")
for c in header.columns:
    print(" -", c)

def resolve_col(columns, aliases):
    lookup = {
        str(c).strip().upper(): c
        for c in columns
    }

    for alias in aliases:
        k = alias.strip().upper()
        if k in lookup:
            return lookup[k]

    return None

COL = {
    "location": resolve_col(
        header.columns,
        ["VRLOCATION", "LOCATION", "Location"]
    ),
    "buyer": resolve_col(
        header.columns,
        ["VRBUYER", "BUYER", "Buyer"]
    ),
    "start_date": resolve_col(
        header.columns,
        [
            "BiWeekly_Start_Date",
            "BIWEEKLY_START_DATE",
            "Biweekly_Start_Date"
        ]
    ),
    "active": resolve_col(
        header.columns,
        ["Active", "ACTIVE"]
    ),
    "fallout": resolve_col(
        header.columns,
        ["Fallout", "FALLOUT"]
    ),
    "dark": resolve_col(
        header.columns,
        ["Dark", "DARK"]
    ),
    "reactivated": resolve_col(
        header.columns,
        ["Reactivated", "REACTIVATED"]
    ),
    "net_new": resolve_col(
        header.columns,
        ["Net_New", "NET_NEW", "Net New"]
    ),
    "days_since_last": resolve_col(
        header.columns,
        [
            "Days_Since_Last_Purchase",
            "DAYS_SINCE_LAST_PURCHASE"
        ]
    ),
    "days_since_prev_loc": resolve_col(
        header.columns,
        [
            "Days_Since_Previous_Location_Purchase",
            "DAYS_SINCE_PREVIOUS_LOCATION_PURCHASE"
        ]
    ),
}

mapping = pd.DataFrame(
    [{"Logical_Field": k, "Resolved_Column": v} for k, v in COL.items()]
)

display(mapping)

required = ["location", "buyer", "start_date"]

missing = [k for k in required if COL[k] is None]

if missing:
    raise ValueError(
        "Missing required buyer columns: "
        + ", ".join(missing)
    )


Buyer columns:
 - VRLOCATION
 - VRBUYER
 - BiWeekly_ID
 - BiWeekly_Start_Date
 - BiWeekly_End_Date
 - first_location_sale_date
 - Last_Purchase_Date
 - Previous_Location_Purchase_Date
 - Days_Since_Previous_Location_Purchase
 - Last_Purchase_Was_Reactivation
 - Days_Since_Last_Purchase
 - Lifecycle_Status
 - Active
 - Fallout
 - Dark
 - Reactivated
 - Net_New


,Logical_Field,Resolved_Column
0,location,VRLOCATION
1,buyer,VRBUYER
2,start_date,BiWeekly_Start_Date
3,active,Active
4,fallout,Fallout
5,dark,Dark
6,reactivated,Reactivated
7,net_new,Net_New
8,days_since_last,Days_Since_Last_Purchase
9,days_since_prev_loc,Days_Since_Previous_Location_Purchase


## 2. Build 2022 buyer baseline in chunks

In [3]:

USECOLS = [
    c for c in [
        COL["location"],
        COL["buyer"],
        COL["start_date"],
        COL["active"],
        COL["fallout"],
        COL["dark"],
        COL["reactivated"],
        COL["net_new"],
        COL["days_since_last"],
        COL["days_since_prev_loc"],
    ]
    if c is not None
]

def clean_text(s):
    return (
        s.astype("string")
        .fillna("")
        .str.strip()
    )

def to_flag(s):
    num = pd.to_numeric(
        s,
        errors="coerce"
    )

    out = pd.Series(
        np.zeros(len(s), dtype=np.int8),
        index=s.index
    )

    numeric_mask = num.notna()

    if numeric_mask.any():
        out.loc[numeric_mask] = (
            num.loc[numeric_mask]
            .gt(0)
            .astype("int8")
        )

    text_mask = ~numeric_mask

    if text_mask.any():
        txt = (
            s.loc[text_mask]
            .astype("string")
            .fillna("")
            .str.strip()
            .str.upper()
        )

        true_values = {
            "1", "Y", "YES", "TRUE",
            "ACTIVE", "FALLOUT", "DARK",
            "REACTIVATED", "NET NEW", "NET_NEW"
        }

        out.loc[text_mask] = (
            txt.isin(true_values)
            .astype("int8")
        )

    return out

partial = []
rows_read = 0
rows_kept = 0

reader = pd.read_csv(
    BUYER_FILE,
    usecols=USECOLS,
    chunksize=CHUNK_SIZE,
    low_memory=True,
    encoding="utf-8",
    encoding_errors="replace"
)

for chunk_num, chunk in enumerate(reader, start=1):

    rows_read += len(chunk)

    dt = pd.to_datetime(
        chunk[COL["start_date"]],
        errors="coerce"
    )

    mask = dt.between(
        BUYER_BASELINE_START,
        BUYER_BASELINE_END
    )

    if not mask.any():
        del chunk, dt, mask
        gc.collect()
        continue

    loc = (
        clean_text(
            chunk.loc[mask, COL["location"]]
        )
        .str.upper()
    )

    buyer_id = clean_text(
        chunk.loc[mask, COL["buyer"]]
    )

    valid = (
        loc.ne("")
        & buyer_id.ne("")
    )

    if not valid.any():
        del chunk, dt, mask, loc, buyer_id, valid
        gc.collect()
        continue

    temp = pd.DataFrame({
        "VRLOCATION": loc.loc[valid].values,
        "VRBUYER": buyer_id.loc[valid].values,
    })

    for logical, outcol in [
        ("active", "Active_Flag"),
        ("fallout", "Fallout_Flag"),
        ("dark", "Dark_Flag"),
        ("reactivated", "Reactivated_Flag"),
        ("net_new", "Net_New_Flag"),
    ]:
        source = COL.get(logical)

        if source is not None:
            vals = chunk.loc[
                mask,
                source
            ].loc[valid]

            temp[outcol] = (
                to_flag(vals)
                .values
            )
        else:
            temp[outcol] = np.int8(0)

    if COL["days_since_last"] is not None:
        vals = chunk.loc[
            mask,
            COL["days_since_last"]
        ].loc[valid]

        temp["Days_Since_Last"] = pd.to_numeric(
            vals,
            errors="coerce"
        ).astype("float32").values
    else:
        temp["Days_Since_Last"] = np.nan

    if COL["days_since_prev_loc"] is not None:
        vals = chunk.loc[
            mask,
            COL["days_since_prev_loc"]
        ].loc[valid]

        temp["Days_Since_Prev_Loc"] = pd.to_numeric(
            vals,
            errors="coerce"
        ).astype("float32").values
    else:
        temp["Days_Since_Prev_Loc"] = np.nan

    agg = (
        temp.groupby(
            "VRLOCATION",
            sort=False
        )
        .agg(
            Buyer_Records=("VRBUYER", "size"),
            Unique_Buyers=("VRBUYER", "nunique"),
            Active_Buyer_Records=("Active_Flag", "sum"),
            Fallout_Buyer_Records=("Fallout_Flag", "sum"),
            Dark_Buyer_Records=("Dark_Flag", "sum"),
            Reactivated_Buyer_Records=("Reactivated_Flag", "sum"),
            Net_New_Buyer_Records=("Net_New_Flag", "sum"),

            Days_Since_Last_Sum=("Days_Since_Last", "sum"),
            Days_Since_Last_Count=("Days_Since_Last", "count"),

            Days_Since_Prev_Loc_Sum=("Days_Since_Prev_Loc", "sum"),
            Days_Since_Prev_Loc_Count=("Days_Since_Prev_Loc", "count"),
        )
        .reset_index()
    )

    partial.append(agg)

    rows_kept += len(temp)

    if chunk_num % 10 == 0:
        print(
            f"Chunks={chunk_num:,} | "
            f"Rows read={rows_read:,} | "
            f"2022 rows kept={rows_kept:,}"
        )

    del chunk, dt, mask, loc, buyer_id, valid, temp, agg
    gc.collect()

print("\nChunk pass complete.")
print("Rows read:", f"{rows_read:,}")
print("2022 rows retained:", f"{rows_kept:,}")


Chunks=10 | Rows read=1,000,000 | 2022 rows kept=334,498
Chunks=20 | Rows read=2,000,000 | 2022 rows kept=668,502
Chunks=30 | Rows read=3,000,000 | 2022 rows kept=1,020,114
Chunks=40 | Rows read=4,000,000 | 2022 rows kept=1,365,164
Chunks=50 | Rows read=5,000,000 | 2022 rows kept=1,704,849
Chunks=60 | Rows read=6,000,000 | 2022 rows kept=2,062,387

Chunk pass complete.
Rows read: 6,679,333
2022 rows retained: 2,289,387


## 3. Combine chunk-level buyer baseline

In [4]:

buyer_base_raw = pd.concat(
    partial,
    ignore_index=True
)

del partial
gc.collect()

buyer_base = (
    buyer_base_raw.groupby(
        "VRLOCATION",
        as_index=False
    )
    .agg(
        Buyer_Records=("Buyer_Records", "sum"),
        Unique_Buyers=("Unique_Buyers", "sum"),
        Active_Buyer_Records=("Active_Buyer_Records", "sum"),
        Fallout_Buyer_Records=("Fallout_Buyer_Records", "sum"),
        Dark_Buyer_Records=("Dark_Buyer_Records", "sum"),
        Reactivated_Buyer_Records=("Reactivated_Buyer_Records", "sum"),
        Net_New_Buyer_Records=("Net_New_Buyer_Records", "sum"),

        Days_Since_Last_Sum=("Days_Since_Last_Sum", "sum"),
        Days_Since_Last_Count=("Days_Since_Last_Count", "sum"),

        Days_Since_Prev_Loc_Sum=("Days_Since_Prev_Loc_Sum", "sum"),
        Days_Since_Prev_Loc_Count=("Days_Since_Prev_Loc_Count", "sum"),
    )
)

del buyer_base_raw
gc.collect()

buyer_base["Active_Rate"] = np.where(
    buyer_base["Buyer_Records"] > 0,
    buyer_base["Active_Buyer_Records"]
    / buyer_base["Buyer_Records"],
    np.nan
)

buyer_base["Fallout_Rate"] = np.where(
    buyer_base["Buyer_Records"] > 0,
    buyer_base["Fallout_Buyer_Records"]
    / buyer_base["Buyer_Records"],
    np.nan
)

buyer_base["Dark_Rate"] = np.where(
    buyer_base["Buyer_Records"] > 0,
    buyer_base["Dark_Buyer_Records"]
    / buyer_base["Buyer_Records"],
    np.nan
)

buyer_base["Reactivated_Rate"] = np.where(
    buyer_base["Buyer_Records"] > 0,
    buyer_base["Reactivated_Buyer_Records"]
    / buyer_base["Buyer_Records"],
    np.nan
)

buyer_base["Net_New_Rate"] = np.where(
    buyer_base["Buyer_Records"] > 0,
    buyer_base["Net_New_Buyer_Records"]
    / buyer_base["Buyer_Records"],
    np.nan
)

buyer_base["Avg_Days_Since_Last_Purchase"] = np.where(
    buyer_base["Days_Since_Last_Count"] > 0,
    buyer_base["Days_Since_Last_Sum"]
    / buyer_base["Days_Since_Last_Count"],
    np.nan
)

buyer_base["Avg_Days_Since_Previous_Location_Purchase"] = np.where(
    buyer_base["Days_Since_Prev_Loc_Count"] > 0,
    buyer_base["Days_Since_Prev_Loc_Sum"]
    / buyer_base["Days_Since_Prev_Loc_Count"],
    np.nan
)

buyer_base.to_csv(
    OUTPUT_DIR / "01_2022_BUYER_LOCATION_BASELINE.csv",
    index=False
)

print("Buyer baseline locations:", len(buyer_base))
display(buyer_base.head())


Buyer baseline locations: 49


,VRLOCATION,Buyer_Records,Unique_Buyers,Active_Buyer_Records,Fallout_Buyer_Records,Dark_Buyer_Records,Reactivated_Buyer_Records,Net_New_Buyer_Records,Days_Since_Last_Sum,Days_Since_Last_Count,Days_Since_Prev_Loc_Sum,Days_Since_Prev_Loc_Count,Active_Rate,Fallout_Rate,Dark_Rate,Reactivated_Rate,Net_New_Rate,Avg_Days_Since_Last_Purchase,Avg_Days_Since_Previous_Location_Purchase
0,ALBANY,6267,348,1283,878,3935,171,5,2150113.0,6267,342146.0,4246,0.204723,0.140099,0.627892,0.027286,0.000798,343.084889,80.580782
1,ATLANTA,83330,4680,7353,6490,67928,1559,192,42537112.0,83330,5508111.0,42704,0.088240,0.077883,0.815169,0.018709,0.002304,510.465763,128.983491
2,AUCTIONINMOTION,55232,3065,3919,4999,45423,891,52,27735630.0,55232,3603860.0,30136,0.070955,0.090509,0.822404,0.016132,0.000941,502.165955,119.586541
3,AUSTIN,68028,3766,7818,7022,51943,1245,62,32214818.0,68028,4823535.0,43550,0.114923,0.103222,0.763553,0.018301,0.000911,473.552331,110.758553
4,BADGERSTATE,41167,2278,3218,3566,33754,629,38,19502388.0,41167,2363487.0,18533,0.078169,0.086623,0.819929,0.015279,0.000923,473.738383,127.528571


## 4. Load 2023+ fee events

In [5]:

events = pd.read_csv(
    DID_FILE,
    low_memory=True
)

events["Change_Date"] = pd.to_datetime(
    events["Corporate_Change_ID"],
    errors="coerce"
)

events["Treated_Location"] = (
    events["Treated_Location"]
    .astype("string")
    .fillna("")
    .str.strip()
    .str.upper()
)

for c in [
    "BUYFEE_CHANGE_AMOUNT",
    "DiD_RPU_Change_Dollar",
    "DiD_Conversion_Change_pp",
    "DiD_Volume_Change_Pct",
]:
    events[c] = pd.to_numeric(
        events[c],
        errors="coerce"
    )

events = events[
    events["Change_Date"].ge(
        FEE_EVENT_START
    )
].copy()

scenario_mask = pd.Series(
    False,
    index=events.index
)

for fee in SCENARIOS:
    scenario_mask |= (
        events[
            "BUYFEE_CHANGE_AMOUNT"
        ]
        .between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    )

events = events[
    scenario_mask
].copy()

events["RPU_Success"] = (
    events["DiD_RPU_Change_Dollar"]
    > RPU_SUCCESS_MIN
)

events["Conversion_Safe"] = (
    events["DiD_Conversion_Change_pp"]
    >= CONVERSION_SAFE_MIN_PP
)

events["Volume_Safe"] = (
    events["DiD_Volume_Change_Pct"]
    >= VOLUME_SAFE_MIN_PCT
)

events["Balanced_Success"] = (
    events["RPU_Success"]
    & events["Conversion_Safe"]
    & events["Volume_Safe"]
)

print("2023+ scenario events:", f"{len(events):,}")
print("Locations:", events["Treated_Location"].nunique())
print("Change dates:", events["Corporate_Change_ID"].nunique())


2023+ scenario events: 810
Locations: 24
Change dates: 29


## 5. Join 2022 buyer baseline to 2023+ fee events

In [6]:

model_df = events.merge(
    buyer_base,
    left_on="Treated_Location",
    right_on="VRLOCATION",
    how="left"
)

model_df["Buyer_Baseline_Available"] = (
    model_df["VRLOCATION"].notna()
)

print(
    "Events with buyer baseline:",
    int(
        model_df[
            "Buyer_Baseline_Available"
        ].sum()
    ),
    "/",
    len(model_df)
)

print(
    "Coverage:",
    round(
        model_df[
            "Buyer_Baseline_Available"
        ].mean()
        * 100,
        2
    ),
    "%"
)

model_df.to_csv(
    OUTPUT_DIR / "02_2023PLUS_EVENTS_WITH_2022_BUYER_BASELINE.csv",
    index=False
)


Events with buyer baseline: 783 / 810
Coverage: 96.67 %


## 6. Compare successful vs unsuccessful events

In [7]:

buyer_features = [
    "Unique_Buyers",
    "Active_Rate",
    "Fallout_Rate",
    "Dark_Rate",
    "Reactivated_Rate",
    "Net_New_Rate",
    "Avg_Days_Since_Last_Purchase",
    "Avg_Days_Since_Previous_Location_Purchase",
]

rows = []

for feature in buyer_features:

    success = pd.to_numeric(
        model_df.loc[
            model_df["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    fail = pd.to_numeric(
        model_df.loc[
            ~model_df["Balanced_Success"],
            feature
        ],
        errors="coerce"
    ).dropna()

    rows.append({
        "Feature": feature,
        "Successful_N": len(success),
        "Failed_N": len(fail),

        "Successful_Median":
            success.median()
            if len(success)
            else np.nan,

        "Failed_Median":
            fail.median()
            if len(fail)
            else np.nan,

        "Median_Difference":
            (
                success.median()
                - fail.median()
            )
            if len(success)
            and len(fail)
            else np.nan,
    })

profile = pd.DataFrame(rows)

profile.to_csv(
    OUTPUT_DIR / "03_BUYER_BASELINE_SUCCESS_VS_FAILURE.csv",
    index=False
)

display(profile)


,Feature,Successful_N,Failed_N,Successful_Median,Failed_Median,Median_Difference
0,Unique_Buyers,236,547,2214.000000,2214.000000,0.000000
1,Active_Rate,236,547,0.149969,0.158712,-0.008743
2,Fallout_Rate,236,547,0.109493,0.102381,0.007112
3,Dark_Rate,236,547,0.720564,0.720564,0.000000
4,Reactivated_Rate,236,547,0.020403,0.019780,0.000623
5,Net_New_Rate,236,547,0.000726,0.000726,0.000000
6,Avg_Days_Since_Last_Purchase,236,547,416.511953,409.340664,7.171289
7,Avg_Days_Since_Previous_Location_Purchase,236,547,102.631653,102.631653,0.000000


## 7. Buyer baseline correlations with fee outcomes

In [8]:

rows = []

for feature in buyer_features:

    x = pd.to_numeric(
        model_df[feature],
        errors="coerce"
    )

    record = {
        "Feature": feature,
        "Events": int(
            x.notna().sum()
        )
    }

    for outcome in [
        "DiD_RPU_Change_Dollar",
        "DiD_Conversion_Change_pp",
        "DiD_Volume_Change_Pct",
    ]:

        y = pd.to_numeric(
            model_df[outcome],
            errors="coerce"
        )

        mask = (
            x.notna()
            & y.notna()
        )

        record[
            f"Corr_with_{outcome}"
        ] = (
            x[mask].corr(
                y[mask]
            )
            if mask.sum() >= 20
            else np.nan
        )

    rows.append(record)

ranking = pd.DataFrame(rows)

corr_cols = [
    c for c in ranking.columns
    if c.startswith("Corr_with_")
]

if corr_cols:
    ranking["Max_Absolute_Correlation"] = (
        ranking[corr_cols]
        .abs()
        .max(axis=1)
    )

    ranking = ranking.sort_values(
        "Max_Absolute_Correlation",
        ascending=False
    )

ranking.to_csv(
    OUTPUT_DIR / "04_BUYER_BASELINE_FEATURE_RANKING.csv",
    index=False
)

display(ranking)


,Feature,Events,Corr_with_DiD_RPU_Change_Dollar,Corr_with_DiD_Conversion_Change_pp,Corr_with_DiD_Volume_Change_Pct,Max_Absolute_Correlation
1,Active_Rate,783,-0.131039,-0.093532,0.007228,0.131039
3,Dark_Rate,783,0.099671,0.057695,-0.005817,0.099671
0,Unique_Buyers,783,-0.097112,0.019045,-0.002270,0.097112
6,Avg_Days_Since_Last_Purchase,783,-0.040219,0.074921,0.032105,0.074921
7,Avg_Days_Since_Previous_Location_Purchase,783,0.036520,0.065011,-0.015907,0.065011
2,Fallout_Rate,783,0.024601,0.048636,-0.000762,0.048636
4,Reactivated_Rate,783,-0.039050,0.003024,0.003766,0.039050
5,Net_New_Rate,783,-0.028118,-0.031770,-0.007271,0.031770


## 8. Scenario-level comparison

In [9]:

scenario_rows = []

for fee in SCENARIOS:

    d = model_df[
        model_df["BUYFEE_CHANGE_AMOUNT"].between(
            fee - SCENARIO_TOLERANCE,
            fee + SCENARIO_TOLERANCE
        )
    ].copy()

    scenario_rows.append({
        "Fee_Increase": fee,
        "Events": len(d),
        "Locations": d["Treated_Location"].nunique(),
        "Corporate_Change_Dates": d["Corporate_Change_ID"].nunique(),

        "Balanced_Success_Rate": d["Balanced_Success"].mean(),
        "RPU_Success_Rate": d["RPU_Success"].mean(),
        "Conversion_Safe_Rate": d["Conversion_Safe"].mean(),
        "Volume_Safe_Rate": d["Volume_Safe"].mean(),

        "Median_RPU_Effect": d["DiD_RPU_Change_Dollar"].median(),
        "Median_Conversion_Effect_pp": d["DiD_Conversion_Change_pp"].median(),
        "Median_Volume_Effect_Pct": d["DiD_Volume_Change_Pct"].median(),

        "Median_2022_Unique_Buyers": d["Unique_Buyers"].median(),
        "Median_2022_Active_Rate": d["Active_Rate"].median(),
        "Median_2022_Fallout_Rate": d["Fallout_Rate"].median(),
        "Median_2022_Dark_Rate": d["Dark_Rate"].median(),
    })

scenario = pd.DataFrame(
    scenario_rows
)

scenario.to_csv(
    OUTPUT_DIR / "05_SCENARIO_WITH_2022_BUYER_BASELINE.csv",
    index=False
)

display(scenario)


,Fee_Increase,Events,Locations,Corporate_Change_Dates,Balanced_Success_Rate,RPU_Success_Rate,Conversion_Safe_Rate,Volume_Safe_Rate,Median_RPU_Effect,Median_Conversion_Effect_pp,Median_Volume_Effect_Pct,Median_2022_Unique_Buyers,Median_2022_Active_Rate,Median_2022_Fallout_Rate,Median_2022_Dark_Rate
0,10,594,16,24,0.289562,0.594276,0.585859,0.611111,7.630647,0.627338,-0.003620,2214.0,0.158712,0.109493,0.720564
1,15,84,14,10,0.297619,0.630952,0.607143,0.654762,10.481982,1.790793,0.044703,1650.0,0.146753,0.099394,0.740384
2,20,133,12,10,0.345865,0.759398,0.586466,0.593985,24.466728,1.658328,-0.025353,1725.0,0.125558,0.102381,0.740070


## 9. Final executive summary

In [10]:

summary = pd.DataFrame([
    {
        "Metric": "Buyer_Baseline_Start",
        "Value": BUYER_BASELINE_START
    },
    {
        "Metric": "Buyer_Baseline_End",
        "Value": BUYER_BASELINE_END
    },
    {
        "Metric": "Buyer_Baseline_Locations",
        "Value": buyer_base["VRLOCATION"].nunique()
    },
    {
        "Metric": "2023plus_Fee_Events",
        "Value": len(model_df)
    },
    {
        "Metric": "2023plus_Events_With_Buyer_Baseline",
        "Value": int(
            model_df["Buyer_Baseline_Available"].sum()
        )
    },
    {
        "Metric": "Buyer_Baseline_Coverage_Rate",
        "Value": model_df["Buyer_Baseline_Available"].mean()
    },
    {
        "Metric": "Fee_Event_Locations",
        "Value": model_df["Treated_Location"].nunique()
    },
    {
        "Metric": "Corporate_Change_Dates",
        "Value": model_df["Corporate_Change_ID"].nunique()
    },
])

summary.to_csv(
    OUTPUT_DIR / "06_EXECUTIVE_SUMMARY.csv",
    index=False
)

print("=" * 80)
print("V12B COMPLETE")
print("=" * 80)

print("\nOutput folder:")
print(OUTPUT_DIR)

display(summary)


V12B COMPLETE

Output folder:
C:\Users\ViditSheth\OneDrive - America's Auto Auction\Finance - Documents\FP&A\Data Analytics\Development\Vidit_Sanbox\Buy Fee Model Phase 2\Rough Area\Volume Attirtion\buy_fee_model_v12b_2022baseline_outputs


,Metric,Value
0,Buyer_Baseline_Start,2022-01-01 00:00:00
1,Buyer_Baseline_End,2022-09-30 00:00:00
2,Buyer_Baseline_Locations,49
3,2023plus_Fee_Events,810
4,2023plus_Events_With_Buyer_Baseline,783
5,Buyer_Baseline_Coverage_Rate,0.966667
6,Fee_Event_Locations,24
7,Corporate_Change_Dates,29



## Files to share back

Please send:

1. `06_EXECUTIVE_SUMMARY.csv`
2. `03_BUYER_BASELINE_SUCCESS_VS_FAILURE.csv`
3. `04_BUYER_BASELINE_FEATURE_RANKING.csv`
4. `05_SCENARIO_WITH_2022_BUYER_BASELINE.csv`

## Interpretation

This notebook does not claim that 2022 buyer behavior directly caused 2023–2026 fee outcomes.

It tests whether **historical buyer health by location** is a useful structural feature for later Buy Fee response.

If these features show useful signal, they can be tested as location-level priors in the next model.
